# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

Welcome to the **LeRobot SmolVLA training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `SmolVLA` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `SmolVLA` policy for 20,000 steps typically takes **about 5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer!

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg, and installs the package in editable mode with train, dataset and smolvla features.

In [1]:
!git clone https://github.com/huggingface/lerobot.git
!apt-get install ffmpeg
!cd lerobot && pip install -e ".[train, dataset, smolvla]"

Cloning into 'lerobot'...
remote: Enumerating objects: 60244, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (31/31), done.
remote: Total 60244 (delta 6), reused 14 (delta 1), pack-reused 60209 (from 1)
Receiving objects: 100% (60244/60244), 249.46 MiB | 26.45 MiB/s, done.
Resolving deltas: 100% (38521/38521), done.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 11.3 

## Weights & Biases login (optional)
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging. This step is optional, you can skip it. If you want to use W&B remember to change `--wandb.enable` to true in the next section.

In [2]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## HF login

To upload your trained model to the hub you need to login with your Hugging Face account.
1. Run the cell below.
2. You will be asked to generate a token in the Hugging Face settings.
3. Select all checkboxes under Repositories when creating the token.
4. Paste the generated token into the command line below.

In [ ]:
!hf auth login

## Start training SmolVLA with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--batch_size=64`: means the model processes 64 training samples in parallel before doing one gradient update. Reduce this number if you have a GPU with low memory.

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this.

In [4]:
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=lerobot/svla_so100_pickplace \
  --policy.device=cuda \
  --batch_size=1 \
  --steps=10000 \
  --output_dir=outputs/train/smolvla_pickplace_10000_step_collab_T4_benchmark \
  --job_name=smolvla_pickplace_10000_step_collab_T4_benchmark \
  --wandb.enable=true \
  --policy.push_to_hub=false \
  --rename_map='{"observation.images.top":"observation.images.camera1","observation.images.wrist":"observation.images.camera2"}' \
  --log_freq=50 \
  --save_freq=5000 \
  --seed=42

config.json: 100% 2.30k/2.30k [00:00<00:00, 6.21MB/s]
INFO 2026-10-12 08:49:14 ot_train.py:580 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'reshard_after_forward': True,
                          'wrap_modules': None},
                 'grad_scaler': {'backoff_factor': 0.5,
                                 'growth_factor': 2.0,
                                 'growth_interval': 2000,
                           

Sometimes after training, you may notice that the model underperforms and cannot solve the task properly. Sometimes this is due to poor data quality, but sometimes the model simply needs more training. To continue training from a previously trained model, use `--policy.pretrained_path=username/path_to_model` and paste the path to the model you trained previously here.

In [6]:
!lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=lerobot/svla_so100_pickplace \
  --policy.device=cuda \
  --batch_size=1 \
  --steps=20000 \
  --output_dir=outputs/train/smolvla_pickplace_20000_step_collab_T4_benchmark \
  --job_name=smolvla_pickplace_20000_step_collab_T4_benchmark \
  --wandb.enable=true \
  --policy.push_to_hub=false \
  --rename_map='{"observation.images.top":"observation.images.camera1","observation.images.wrist":"observation.images.camera2"}' \
  --log_freq=100 \
  --save_freq=20000 \
  --seed=42

INFO 2026-10-12 10:12:28 ot_train.py:580 {'accelerator': {'activation_checkpointing': {'mode': <ActivationCheckpointingMode.NONE: 'none'>},
                 'compile': {'backend': 'inductor',
                             'enabled': False,
                             'mode': None,
                             'regional': True},
                 'ddp': {'find_unused_parameters': True,
                         'gradient_as_bucket_view': False,
                         'static_graph': False},
                 'fsdp': {'cpu_offload': False,
                          'ignored_modules': None,
                          'min_num_params': None,
                          'reshard_after_forward': True,
                          'wrap_modules': None},
                 'grad_scaler': {'backoff_factor': 0.5,
                                 'growth_factor': 2.0,
                                 'growth_interval': 2000,
                                 'init_scale': 65536.0},
                 'gradie

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")

print("Google Drive accessible:", drive.exists())
print("\nFolders and files:")
for item in sorted(drive.iterdir()):
    print(("📁 " if item.is_dir() else "📄 ") + item.name)

Google Drive accessible: True

Folders and files:
📄  Shinu Shaju - Seminar EIM: Research Trends in Systems and Control (SoSe 26).gslides
📄 1_A_Shinu_Shaju.docx
📄 240912_Antrag_auf_Anerkennung Shinu Shaju.pdf
📄 2_A_Shinu_Shaju.docx
📄 2_Sanzina_Hossain_V1.docx
📄 3_A_Shinu_Shaju.docx
📄 3_David_Nowak_V1.docx
📄 5_Shinu_Shaju_v2.docx
📄 A1 Shinu Shaju.pdf
📁 Accomodation
📄 BD25_Project_F4_B: Real-Time Traffic Monitoring Architecture Diagram.gdoc
📄 Certificate of Insurance.pdf
📄 Copy of  Seminar EIM: Research Trends in Systems and Control (SoSe 26).gslides
📄 Copy of Cover Letter - Working Student 1KOMMA5°.docx
📄 Copy of Reinforcement Learning WiSe 25 26.gslides
📄 Copy of Shinu_Shaju_641425_Research_Project_IDAC.gdoc
📄 Cover Letter - Working Student 1KOMMA5°.docx
📄 Cyber Secuirty PBL Report Draft.gdoc
📄 Cyber Security Data Science.gdoc
📄 DSF-Application_for_Research_Project-Shinu_Shaju_641425.docx
📁 Documents
📄 E-Aadhaar_Shinu.pdf
📄 Erste_Miete_Shinu_Shaju_2025-03-13_1215.pdf
📄 Hausordnung.pdf
📄

In [11]:
!find /content -type d -path "*/checkpoints/010000/pretrained_model" 2>/dev/null

/content/outputs/train/smolvla_pickplace_10000_step_collab_T4_benchmark/checkpoints/010000/pretrained_model


In [12]:
from pathlib import Path
import shutil

source = Path(
    "/content/outputs/train/"
    "smolvla_pickplace_10000_step_collab_T4_benchmark/"
    "checkpoints/010000/pretrained_model"
)

destination = Path(
    "/content/drive/MyDrive/smolvla/10k/pretrained_model"
)

if not source.exists():
    raise FileNotFoundError(f"Checkpoint not found: {source}")

destination.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(source, destination, dirs_exist_ok=True)

print("Checkpoint copied:", destination.exists())
print("Files copied:", sum(1 for p in destination.rglob("*") if p.is_file()))

for file in sorted(destination.iterdir()):
    print(file.name)

Checkpoint copied: True
Files copied: 10
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
policy_preprocessor.json
policy_preprocessor_step_5_normalizer_processor.safetensors
tokenizer
train_config.json


In [13]:
from pathlib import Path

for path in Path("/content/outputs/train").glob(
    "**/checkpoints/020000/pretrained_model"
):
    print("20k checkpoint found:", path)

20k checkpoint found: /content/outputs/train/smolvla_pickplace_20000_step_collab_T4_benchmark/checkpoints/020000/pretrained_model


In [14]:
from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive if needed
drive.mount("/content/drive")

# Source: your existing 20k checkpoint
source = Path(
    "/content/outputs/train/"
    "smolvla_pickplace_20000_step_collab_T4_benchmark/"
    "checkpoints/020000/pretrained_model"
)

# Destination: Google Drive
destination = Path(
    "/content/drive/MyDrive/smolvla/20k/pretrained_model"
)

if not source.exists():
    raise FileNotFoundError(f"Checkpoint not found: {source}")

destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Checkpoint copied successfully!")
print("Destination:", destination)
print("Number of files:", sum(
    1 for p in destination.rglob("*") if p.is_file()
))

for item in sorted(destination.iterdir()):
    print(item.name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint copied successfully!
Destination: /content/drive/MyDrive/smolvla/20k/pretrained_model
Number of files: 10
config.json
model.safetensors
policy_postprocessor.json
policy_postprocessor_step_0_unnormalizer_processor.safetensors
policy_preprocessor.json
policy_preprocessor_step_5_normalizer_processor.safetensors
tokenizer
train_config.json
